In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Extracción y Normalización del Perfil de Generación Solar Fotovoltaica

Este script procesa las series temporales de generación renovable obtenidas de 
la base de datos meteorológica Renewables.ninja para la ubicación de Sevilla.
Realiza la ingesta de datos, el filtrado del día de estudio y la normalización 
en por unidad (p.u.) asumiendo una instalación base de 1 kW de capacidad nominal.
Esta curva se asigna a los nodos del clúster como vector de entrada para el optimizador.

Autor: Alberto Zafra Muñoz
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional

# ==========================================
# 1. CONSTANTES Y CONFIGURACIÓN GENERAL
# ==========================================
ARCHIVO_ENTRADA = "Generación_Fotovoltáica_Sevilla_2016.csv"
ARCHIVO_SALIDA = "Generacion_P_G_Limpios.csv"
FECHA_ESTUDIO = '2016-04-01'

# Parámetros de lectura específicos para el formato de Renewables.ninja
FILAS_METADATOS_OMITIR = 3 

# ==========================================
# 2. PROCESAMIENTO DE DATOS (ETL)
# ==========================================
def procesar_perfil_solar(ruta_csv: str, fecha_objetivo: str) -> Optional[pd.DataFrame]:
    """
    Lee el archivo en bruto de Renewables.ninja, extrae la serie temporal
    del día objetivo y prepara la curva base unitaria.

    Parámetros:
        ruta_csv (str): Ruta al archivo CSV con los datos meteorológicos/solares.
        fecha_objetivo (str): Fecha de extracción en formato 'YYYY-MM-DD'.

    Retorna:
        pd.DataFrame: DataFrame con el perfil de generación horario normalizado.
                      Devuelve None en caso de error de lectura.
    """
    print(f"[*] Iniciando ingesta de datos solares desde: {ruta_csv}")
    
    if not os.path.exists(ruta_csv):
        print(f"[ERROR] No se encuentra el dataset fuente '{ruta_csv}'.")
        return None

    try:
        # Lectura omitiendo los metadatos iniciales del simulador
        df = pd.read_csv(ruta_csv, skiprows=FILAS_METADATOS_OMITIR)
        
        # Identificación dinámica de la columna temporal (depende de la versión exportada)
        col_tiempo = 'local_time' if 'local_time' in df.columns else 'time'
        df[col_tiempo] = pd.to_datetime(df[col_tiempo])
        
        # Filtrado del horizonte temporal de estudio
        fecha_filtro = pd.to_datetime(fecha_objetivo).date()
        df_dia = df[df[col_tiempo].dt.date == fecha_filtro].copy()

        if df_dia.empty:
            print(f"[ERROR] No hay registros solares para la fecha {fecha_objetivo}.")
            return None

        print(f"[*] Datos extraídos para el {fecha_objetivo}: {len(df_dia)} registros horarios.")

        # Transformación del vector temporal
        df_dia['Hora'] = df_dia[col_tiempo].dt.hour
        
        # Normalización: Para una planta de 1kW, la producción (kWh) equivale al valor en p.u.
        # Se replica el perfil normalizado para los diferentes nodos del clúster
        df_dia['U1_Solar_pu'] = df_dia['electricity']
        df_dia['U2_Solar_pu'] = df_dia['electricity']
        df_dia['U3_Solar_pu'] = df_dia['electricity']

        # Consolidación del DataFrame final
        columnas_interes = ['Hora', 'U1_Solar_pu', 'U2_Solar_pu', 'U3_Solar_pu']
        df_horario = df_dia[columnas_interes].reset_index(drop=True)

        print("[*] Normalización de perfiles completada con éxito.")
        return df_horario

    except Exception as e:
        print(f"[ERROR INESPERADO] Fallo durante el procesamiento solar: {e}")
        return None

# ==========================================
# 3. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_generacion_solar(df: pd.DataFrame):
    """
    Genera la figura del perfil de radiación solar transformado en potencia base.
    """
    plt.figure(figsize=(10, 5))
    plt.style.use('seaborn-v0_8-whitegrid')

    # Trazado de la campana de generación diurna
    plt.plot(df['Hora'], df['U1_Solar_pu'], 
             color='#ffbf00', marker='o', linewidth=2.5, markersize=6, 
             label='Generación Solar Unitaria ($P_{u,t}^{G,BASE}$)')
    
    # Sombreado del área de energía total disponible
    plt.fill_between(df['Hora'], df['U1_Solar_pu'], 
                     color='#ffbf00', alpha=0.25)

    # Formateo y Estilos de la Figura
    plt.title('Perfil Normalizado de Generación Solar Fotovoltaica Disponible', 
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Hora del día ($t$)', fontsize=12)
    plt.ylabel('Potencia Base Unitaria (p.u.)', fontsize=12)
    
    # Configuración de ejes
    plt.xticks(range(0, 24))
    plt.xlim(0, 23)
    plt.ylim(0, df['U1_Solar_pu'].max() * 1.15)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper right', frameon=True, shadow=True, fontsize=11)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    df_resultado = procesar_perfil_solar(ARCHIVO_ENTRADA, FECHA_ESTUDIO)
    
    if df_resultado is not None:
        # Guardar dataset limpio como input del optimizador Pyomo
        df_resultado.to_csv(ARCHIVO_SALIDA, index=False)
        print(f"[*] Perfiles solares exportados satisfactoriamente a '{ARCHIVO_SALIDA}'.\n")
        
        # Mostrar preview en consola (primeras 24h)
        print("==================================================================")
        print(" VISTA PREVIA: PERFIL SOLAR NORMALIZADO (p.u.)")
        print("==================================================================")
        print(df_resultado.head(24).to_string(index=False))
        print("==================================================================\n")
        
        print("[*] Generando modelo visual...")
        visualizar_generacion_solar(df_resultado)